# Практика: признаки строки заказа


Работаем как команда CRM маркетплейса: из трёх связанных таблиц нужно получить
объяснимые признаки, а не просто добиться вывода без ошибки. Перед каждой
операцией сформулируйте единицу наблюдения, ключ соединения и ожидаемое число
строк. После операции прочитайте assert как исполняемый контракт.

Сначала сделайте минимальный рабочий вариант, затем проверьте его на данных и
только после этого интерпретируйте результат. Не вводите метку churn: в этом
модуле мы строим и проверяем признаки, но не обучаем модель оттока.


**Центральная идея:** Признак должен быть вычислим в заявленный момент, сохранять форму данных и иметь проверяемый диапазон.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

def find_csv(name):
    for path in (
        Path(name),
        Path("../") / name,
        Path("../../data") / name,
        Path("../data") / name,
        Path("../../../data") / name,
    ):
        if path.exists():
            return path.resolve()
    return "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_05_shop_feature_engineering/data/" + name

orders = pd.read_csv(find_csv("orders_slim.csv"), parse_dates=["order_purchase_timestamp", "order_delivered_customer_date"])
customers = pd.read_csv(find_csv("customers_slim.csv"))
payments = pd.read_csv(find_csv("payments_slim.csv"))
assert len(orders) and len(customers) and len(payments)
assert orders["order_id"].is_unique and customers["customer_id"].is_unique
print(f"orders={len(orders)}, customers={len(customers)}, payments={len(payments)}")


## 1. Календарные признаки

Создайте month и weekday векторно.

**Зачем:** Признак должен быть вычислим в заявленный момент, сохранять форму данных и иметь проверяемый диапазон. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
work=orders.copy()
work["order_month"]=None; work["weekday"]=None  # TODO
assert work["order_month"].between(1,12).all()
assert work["weekday"].between(0,6).all()


## 2. Выходной день через lambda

Сделайте is_weekend из weekday.

**Зачем:** Признак должен быть вычислим в заявленный момент, сохранять форму данных и иметь проверяемый диапазон. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
work["is_weekend"]=None  # TODO
assert set(work["is_weekend"])=={0,1}
assert (work["is_weekend"] == work["weekday"].isin([5,6]).astype(int)).all()


## 3. Join с оплатой

Добавьте тип и сумму, используя validate.

**Зачем:** Признак должен быть вычислим в заявленный момент, сохранять форму данных и иметь проверяемый диапазон. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
joined=None  # TODO
assert len(joined)==len(work) and joined["order_id"].is_unique
assert joined["payment_value"].notna().all()


## 4. Срок доставки

Создайте days_to_deliver векторно.

**Зачем:** Признак должен быть вычислим в заявленный момент, сохранять форму данных и иметь проверяемый диапазон. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
joined["days_to_deliver"]=None  # TODO
assert joined["days_to_deliver"].dropna().ge(0).all()
assert joined["days_to_deliver"].notna().sum()>3000


## 5. Порог выброса

Найдите p99 и выделите строки не ниже него; учитывайте совпадения на границе.

**Зачем:** Признак должен быть вычислим в заявленный момент, сохранять форму данных и иметь проверяемый диапазон. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
p99=None; outlier_mask=None  # TODO
assert p99>0 and outlier_mask.dtype==bool
assert 1 <= int(outlier_mask.sum()) < len(joined)


## 6. Категория срока

Напишите функцию: missing, fast <=7, normal <=14, slow.

**Зачем:** Признак должен быть вычислим в заявленный момент, сохранять форму данных и иметь проверяемый диапазон. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
def delivery_band(days):
    # TODO
    ...
joined["delivery_band"]=joined["days_to_deliver"].apply(delivery_band)
assert {"fast","normal","slow"} <= set(joined["delivery_band"]) <= {"missing","fast","normal","slow"}


## 7. Проверка формы и ключа

Соберите quality_checks из четырёх булевых проверок.

**Зачем:** Признак должен быть вычислим в заявленный момент, сохранять форму данных и иметь проверяемый диапазон. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
quality_checks={"rows":None,"unique_key":None,"payment_complete":None,"dates_nonnegative":None}  # TODO
assert set(quality_checks.values())=={True}


## 8. Риск утечки времени

Объясните, когда delivery-признак недоступен.

**Зачем:** Признак должен быть вычислим в заявленный момент, сохранять форму данных и иметь проверяемый диапазон. Зафиксируйте ожидаемую форму результата до запуска. Если assert падает, сравните единицу наблюдения до и после операции; не удаляйте проверку и не подгоняйте константу под случайный вывод.

In [ ]:
LEAKAGE_NOTE=""  # TODO
assert len(LEAKAGE_NOTE)>=240
assert all(w in LEAKAGE_NOTE.lower() for w in ["момент","достав","модель"])
